# Single-Stock Phased Analysis

**Version:** 1.0 — 2026-03-23  
**Module:** `Analysis/stock_analysis.py`  
**Provider:** `fmp_cached` (sole provider — no fallback)  
**Spec:** `Analysis/docs/PHASED_ANALYSIS_MASTER_PLAN.md`

This notebook executes the full 7-phase investment analysis pipeline by importing from `stock_analysis.py`.  
All heavy computation, API calls, and derived metrics live in the module — the notebook is display-only.

---

## How to use
1. Set `SYMBOL` in **Cell 2** (e.g. `"MSFT"`, `"AAPL"`, `"NVDA"`).
2. Run all cells in order (`Kernel → Restart & Run All`).
3. Review the Phase 7 composite score and action label at the bottom.

| Phase | Name | Gate |
|-------|------|------|
| 1 | Company Profile & Business Quality | Business understandable + data coverage OK |
| 2 | 5-Year Fundamental Analysis | Weighted score ≥ 3.5 / 5.0 |
| 3 | Technical Analysis & Trade Timing | ≥ 6 of 11 bullish conditions |
| 4 | Valuation & Fair Value | 2+ lenses agree + MOS computed |
| 5 | Risk & Portfolio Context | Position sized + portfolio fit classified |
| 6 | Market Segment & Peer Relative | Relative score ≥ 3.5 / 5.0 |
| 7 | Decision, Execution & Monitoring | Composite score → action label |

---
## Cell 1 — Imports & Environment

In [1]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT     = Path(os.getcwd()).resolve()
ANALYSIS_DIR  = REPO_ROOT / "Analysis"
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

# ── Environment / credentials ─────────────────────────────────────────────
load_dotenv(REPO_ROOT / ".env", override=False)   # .env in repo root

# ── OpenBB ────────────────────────────────────────────────────────────────
from openbb import obb   # credentials auto-loaded from user_settings.json

# ── Analysis module ───────────────────────────────────────────────────────
from stock_analysis import (
    AnalysisConfig,
    phase1_company_profile,
    phase2_fundamentals,
    phase3_technicals,
    phase4_valuation,
    phase5_risk,
    phase6_peer_relative,
    phase7_decision,
)

# ── Display helpers ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")

Environment ready.


---
## Cell 2 — Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CHANGE SYMBOL HERE                                                  ║
# ╚══════════════════════════════════════════════════════════════════════╝
SYMBOL = "MSFT"

cfg = AnalysisConfig(
    symbol    = SYMBOL,
    benchmark = "SPY",
    provider  = "fmp_cached",
    enforce_gates = False,  # Set True to stop pipeline on gate failure
)

print(f"Analysis target : {cfg.symbol}")
print(f"Benchmark       : {cfg.benchmark}")
print(f"Provider        : {cfg.provider}")
print(f"Enforce gates   : {cfg.enforce_gates}")
print(f"Fundamental window : {cfg.start_fundamentals}  →  {cfg.end_date}")
print(f"Technical window   : {cfg.start_technicals}  →  {cfg.end_date}")

---
# Phase 1 — Company Profile & Business Quality Screen

> **Objective:** Build a business-first view before touching charts.  
> *"Is this a business we should even spend time analysing?"*
>
> **Gate:** Business understandable + no unpriced existential risk + sufficient data coverage.

**Endpoints used (all `fmp_cached`):**  
`equity.profile` · `equity.price.quote` · `equity.fundamental.metrics` · `equity.compare.peers`  
`equity.fundamental.revenue_per_geography` · `equity.ownership.insider_trading`  
`equity.ownership.institutional` · `equity.estimates.price_target`

In [3]:
p1 = phase1_company_profile(cfg)
print(f"Gate passed: {p1.gate_passed}  |  {p1.gate_notes}")

Failed to fetch data from FMP for InstitutionalOwnership: Unauthorized FMP request -> 402 -> Restricted Endpoint: This endpoint is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"User is unable to access this feature - https://bit.ly/yahoo-finance-api-feedback"}}}


Gate passed: True  |  OK


In [13]:
p1.institutional_df

""


In [ ]:
# ── Company overview ───────────────────────────────────────────────────────
overview = {
    "Symbol":       cfg.symbol,
    "Sector":       p1.sector,
    "Industry":     p1.industry,
    "Market Cap":   f"${p1.market_cap/1e9:.1f}B",
    "Peers":        ", ".join(p1.peers[:8]),
    "Gate":         "✅ PASS" if p1.gate_passed else "❌ FAIL",
}
pd.DataFrame(overview.items(), columns=["Field", "Value"]).set_index("Field")

In [ ]:
# ── Profile summary ───────────────────────────────────────────────────────
if not p1.profile_df.empty:
    display_cols = [c for c in ["company_name","sector","industry","country",
                                 "full_time_employees","description"] if c in p1.profile_df.columns]
    display(p1.profile_df[display_cols].T if display_cols else p1.profile_df.head())

In [ ]:
# ── Geographic revenue breakdown ──────────────────────────────────────────
if not p1.geo_df.empty:
    print("Revenue by Geography (latest year):")
    display(p1.geo_df.head(10))
else:
    print("Geographic revenue data not available for this ticker.")

In [ ]:
# ── Analyst price targets ─────────────────────────────────────────────────
if not p1.price_targets_df.empty:
    display_cols = [c for c in ["published_date","analyst_company","price_target",
                                 "adj_price_target","action"] if c in p1.price_targets_df.columns]
    display(p1.price_targets_df[display_cols].head(10) if display_cols else p1.price_targets_df.head(10))
else:
    print("Price target data not available.")

### Phase 1 Commentary

**Questions to answer before proceeding:**
1. What exactly does the company sell, and to whom?
2. What moat exists — brand, switching costs, patents, network effects, scale?
3. What can break the thesis in 12 months?
4. Is management capital allocation shareholder-friendly (buybacks / dividends / ROIC discipline)?
5. Which external factors dominate outcomes (rates, commodities, regulation)?

> ✏️ *Add your qualitative notes here after reviewing the data above.*

---
# Phase 2 — 5-Year Fundamental Analysis

> **Objective:** Measure business quality, durability, and improvement trend over 5 years.  
> Includes earnings quality (Accruals/Sloan), DuPont decomposition, Novy-Marx profitability, operating leverage.
>
> **Gate:** Weighted composite score ≥ 3.5 / 5.0 · Accruals ratio < 20% · No hard-floor category ≤ 1.5

**Endpoints used (all `fmp_cached`):**  
`equity.fundamental.income` · `equity.fundamental.balance` · `equity.fundamental.cash` · `equity.fundamental.ratios`

In [ ]:
p2 = phase2_fundamentals(cfg)
print(f"Phase 2 composite score : {p2.score:.3f} / 5.0")
print(f"Gate passed             : {p2.gate_passed}  |  {p2.gate_notes}")

In [ ]:
# ── KPI summary table ─────────────────────────────────────────────────────
print("=== Phase 2 KPI Summary ===")
kpi = p2.kpi_df.T.rename(columns={0: "Value"})
kpi["Value"] = kpi["Value"].apply(lambda x: f"{x:.4f}" if not (isinstance(x, float) and np.isnan(x)) else "n/a")
display(kpi)

In [ ]:
# ── Earnings quality spotlight ────────────────────────────────────────────
import math
accruals  = p2.accruals_ratio
gp_ratio  = p2.gross_profitability
op_lev    = p2.operating_leverage
dilution  = p2.dilution_5y

def fmt(v, pct=True):
    if isinstance(v, float) and math.isnan(v): return "n/a"
    return f"{v:.2%}" if pct else f"{v:.3f}"

quality_summary = [
    ("Accruals Ratio (Sloan)",        fmt(accruals),   "< 10% = strong   |  > 20% = concern"),
    ("Gross Profitability (Novy-Marx)",fmt(gp_ratio),   "> 33% = quality anchor"),
    ("Operating Leverage (5Y avg)",    fmt(op_lev, False), "1–1.5x = moderate  |  >2.5x = high risk"),
    ("Share Dilution 5Y",              fmt(dilution),   "Negative = buyback program"),
]
display(pd.DataFrame(quality_summary, columns=["Metric", "Value", "Guidance"]))

In [ ]:
# ── DuPont ROE decomposition ──────────────────────────────────────────────
if not p2.roe_decomp_df.empty:
    print("=== DuPont ROE Decomposition (5Y) ===")
    display(p2.roe_decomp_df.round(4))
else:
    print("DuPont decomposition not available (missing balance sheet columns).")

In [ ]:
# ── Margin trend chart ────────────────────────────────────────────────────
income_df  = p2.income_df
rev_col    = next((c for c in ["revenue","total_revenue"] if c in income_df.columns), None)
gp_col     = next((c for c in ["gross_profit","grossProfit"] if c in income_df.columns), None)
oi_col     = next((c for c in ["operating_income","operatingIncome"] if c in income_df.columns), None)
ni_col     = next((c for c in ["net_income","netIncome"] if c in income_df.columns), None)

if rev_col and len(income_df) > 1:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = range(len(income_df))
    labels = income_df.get("date", income_df.index).astype(str).tolist()
    if gp_col:
        ax.plot(x, income_df[gp_col] / income_df[rev_col] * 100, "o-", label="Gross Margin %")
    if oi_col:
        ax.plot(x, income_df[oi_col] / income_df[rev_col] * 100, "s-", label="Operating Margin %")
    if ni_col:
        ax.plot(x, income_df[ni_col] / income_df[rev_col] * 100, "^-", label="Net Margin %")
    ax.set_xticks(list(x)); ax.set_xticklabels([l[:4] for l in labels], rotation=45)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_title(f"{cfg.symbol} — 5-Year Margin Trends"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("Insufficient data for margin chart.")

### Phase 2 Commentary

**Scorecard categories (v1.1 — 6 blocks):**

| Category | Weight | Signal to look for |
|---|---:|---|
| Growth quality | 20% | Revenue / EPS / FCF CAGR all positive and accelerating |
| Profitability | 20% | Gross margin stable/rising; ROIC > WACC |
| Capital efficiency | 20% | DuPont margin-driven, not leverage-driven; buyback program |
| Balance sheet safety | 15% | Net Debt/EBITDA < 3×; interest coverage > 4× |
| Cash flow quality | 15% | CFO/NI > 0.9; accruals ratio < 10% |
| Structural efficiency | 10% | Operating leverage < 2×; SG&A ratio declining |

> ✏️ *Add your fundamental assessment notes here.*

---
# Phase 3 — Technical Analysis & Trade Timing

> **Objective:** Identify *when* price action, volume, and market structure align with the fundamental thesis.  
> Not prediction — confirmation that conditions favour entry.
>
> **Gate:** ≥ 6 of 11 bullish conditions · No hard override triggered

**Endpoint used:** `equity.price.historical` (1yr daily) + `equity.calendar.earnings`  
**Indicator stack:** SMA 50/200 · ADX · RSI · MACD · OBV · Volume ratio · VWAP · Ichimoku · ROC(20/60/120) · CMF · Weekly confluence · Fibonacci

In [ ]:
p3 = phase3_technicals(cfg)
print(f"Bullish conditions  : {p3.bullish_count} / {len(p3.signals)}  (gate = 6)")
print(f"Entry quality       : {p3.entry_quality}")
print(f"Days to earnings    : {p3.days_to_earnings}  |  Safe window: {p3.earnings_safe_window}")
print(f"Weekly trend        : {'✅ Bullish' if p3.weekly_trend_bullish else '❌ Bearish'}")
print(f"ATR(14)             : {p3.atr:.4f}")
print(f"Gate passed         : {p3.gate_passed}  |  {p3.gate_notes}")

In [ ]:
# ── Signal table (all conditions) ─────────────────────────────────────────
signal_desc = {
    "sma_golden_cross":    "SMA50 > SMA200 (Golden Cross)",
    "adx_trending":        "ADX(14) ≥ 20 (trend exists)",
    "rsi_pullback":        "RSI(14) in 40–60 (healthy pullback zone)",
    "macd_bullish":        "MACD line > signal line",
    "obv_rising":          "OBV slope positive (20D)",
    "volume_ratio_normal": "Volume ratio ≤ 2.5× (not parabolic)",
    "above_vwap":          "Close > cumulative VWAP (institutional positive)",
    "above_cloud":         "Close > Ichimoku cloud top",
    "momentum_positive":   "ROC-60 and ROC-120 both > 0",
    "cmf_positive":        "Chaikin Money Flow(21) > 0",
    "weekly_trend_bullish":"Weekly close > SMA20 AND weekly ADX > 20",
    "stochastic_bullish":  "Stochastic %K > %D and %K < 80 (not overbought, trending up)",
    "bb_squeeze_breakout": "BB width in bottom 20th %ile AND close > upper band (squeeze breakout)",
}
sig_df = pd.DataFrame([
    {"Signal": signal_desc.get(k, k), "Result": "✅" if v else "❌"}
    for k, v in p3.signals.items()
])
sig_df["#"] = range(1, len(sig_df)+1)
display(sig_df.set_index("#")[["Signal", "Result"]])
print(f"\nTotal bullish: {p3.bullish_count}/{len(p3.signals)}  →  Entry quality: {p3.entry_quality}")

In [ ]:
# ── Fibonacci retracement levels ──────────────────────────────────────────
current_price = float(p3.price_df["close"].iloc[-1])
print(f"Current price: ${current_price:.2f}")
print("\nFibonacci Retracement Levels (1-year swing):")
for level, price in p3.fib_levels.items():
    diff = (current_price - price) / current_price
    status = "← PRICE HERE" if abs(diff) < 0.015 else ("above" if diff > 0 else "below")
    print(f"  {level:6s}  ${price:>9.2f}   ({diff:+.1%} from current)  {status}")

In [ ]:
# ── Price chart with key indicators ──────────────────────────────────────
price_df = p3.price_df.copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1, 1]})

# Panel 1: Price + SMAs + VWAP + Bollinger Bands
ax1 = axes[0]
ax1.plot(price_df.index, price_df["close"],  color="#1f77b4", lw=1.5, label="Close")
if "sma_50"  in price_df.columns: ax1.plot(price_df.index, price_df["sma_50"],  "--", color="orange", lw=1, label="SMA50")
if "sma_200" in price_df.columns: ax1.plot(price_df.index, price_df["sma_200"], "--", color="red",    lw=1, label="SMA200")
if "vwap"    in price_df.columns: ax1.plot(price_df.index, price_df["vwap"],    "-.", color="purple", lw=1, label="VWAP")
if "bb_upper" in price_df.columns:
    ax1.fill_between(price_df.index, price_df["bb_lower"], price_df["bb_upper"],
                     alpha=0.08, color="grey", label="BB(20,2)")
# Fibonacci levels
colors_fib = ["#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
for (lvl, price_lvl), col in zip(p3.fib_levels.items(), colors_fib):
    ax1.axhline(price_lvl, color=col, lw=0.6, linestyle=":", alpha=0.7)
    ax1.text(price_df.index[-1], price_lvl, f" {lvl}", fontsize=6, color=col, va="center")
ax1.set_title(f"{cfg.symbol} — Technical Overview  |  {p3.bullish_count}/11 bullish  |  Entry: {p3.entry_quality}")
ax1.legend(fontsize=7, loc="upper left"); ax1.grid(alpha=0.2)

# Panel 2: RSI
ax2 = axes[1]
if "rsi" in price_df.columns:
    ax2.plot(price_df.index, price_df["rsi"], color="#17becf", lw=1)
    ax2.axhline(70, color="red",   lw=0.7, linestyle="--")
    ax2.axhline(30, color="green", lw=0.7, linestyle="--")
    ax2.axhspan(40, 60, alpha=0.07, color="green")
    ax2.set_ylabel("RSI(14)"); ax2.set_ylim(0, 100); ax2.grid(alpha=0.2)

# Panel 3: MACD histogram
ax3 = axes[2]
if "macd_hist" in price_df.columns:
    hist = price_df["macd_hist"]
    colors_hist = ["#2ca02c" if v >= 0 else "#d62728" for v in hist]
    ax3.bar(price_df.index, hist, color=colors_hist, width=1, alpha=0.7)
    ax3.axhline(0, color="black", lw=0.5)
    ax3.set_ylabel("MACD Hist"); ax3.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ── Volume + OBV + CMF chart ──────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True,
                          gridspec_kw={"height_ratios": [2, 1, 1]})

ax1 = axes[0]
ax1.bar(price_df.index, price_df["volume"] / 1e6, color="#aec7e8", width=1, alpha=0.7)
ax1.set_ylabel("Volume (M)"); ax1.set_title(f"{cfg.symbol} — Volume Confirmation"); ax1.grid(alpha=0.2)

ax2 = axes[1]
if "obv" in price_df.columns:
    ax2.plot(price_df.index, price_df["obv"] / 1e6, color="#ff7f0e", lw=1)
    ax2.set_ylabel("OBV (M)"); ax2.grid(alpha=0.2)

ax3 = axes[2]
if "cmf_21" in price_df.columns:
    cmf = price_df["cmf_21"]
    ax3.bar(price_df.index, cmf, color=["#2ca02c" if v >= 0 else "#d62728" for v in cmf], width=1, alpha=0.7)
    ax3.axhline(0, color="black", lw=0.5)
    ax3.set_ylabel("CMF(21)"); ax3.grid(alpha=0.2)

plt.tight_layout()
plt.show()

### Phase 3 Commentary

**Hard overrides to check:**
- `earnings_safe_window == False` → cap technical block at 2.5/5 in Phase 7
- `weekly_trend_bullish == False` → cap technical block at 2.0/5 in Phase 7

**Entry quality tiers:**
- **High Conviction**: VWAP ✅, Ichimoku cloud ✅, ROC momentum ✅, weekly trend ✅ — all four aligned
- **Standard**: ≥ 6/11 conditions but not all four high-conviction signals
- **Cautious**: < 6/11 — do not enter full size

> ✏️ *Add your technical assessment notes here.*

---
# Phase 4 — Valuation & Fair Value Estimation

> **Objective:** Convert business quality into a price decision: cheap, fair, or expensive —  
> then confirm that the technical setup supports entering at current price.
>
> **Four lenses:** Relative multiples · DCF (5Y explicit + terminal) · Reverse-DCF · Quality overlays  
> **Gate:** ≥ 2 lenses agree on direction · MOS computed · Altman Z > 1.81

**Data reused from Phase 2** — no additional API calls.

In [ ]:
p4 = phase4_valuation(cfg, p2, p3)
print(f"Valuation verdict       : {p4.valuation_verdict}")
print(f"DCF fair value          : ${p4.dcf_fair_value:.2f}")
print(f"Current price           : ${current_price:.2f}")
print(f"Margin of Safety (MOS)  : {p4.margin_of_safety:.1%}" if not (isinstance(p4.margin_of_safety, float) and np.isnan(p4.margin_of_safety)) else "Margin of Safety: n/a")
print(f"Implied growth (Reverse-DCF): {p4.implied_growth:.1%}" if not (isinstance(p4.implied_growth, float) and np.isnan(p4.implied_growth)) else "Implied growth: n/a")
print(f"ROIC − WACC spread      : {p4.roic_wacc_spread:.2%}" if not (isinstance(p4.roic_wacc_spread, float) and np.isnan(p4.roic_wacc_spread)) else "ROIC-WACC spread: n/a")
print(f"Piotroski F-Score       : {p4.piotroski:.0f} / 9" if not (isinstance(p4.piotroski, float) and np.isnan(p4.piotroski)) else "Piotroski: n/a")
print(f"Altman Z-Score          : {p4.altman:.2f}" if not (isinstance(p4.altman, float) and np.isnan(p4.altman)) else "Altman Z: n/a")
print(f"Entry recommendation    : {p4.entry_recommendation}")
print(f"Gate passed             : {p4.gate_passed}  |  {p4.gate_notes}")

In [ ]:
# ── Multiples table ───────────────────────────────────────────────────────
print("=== Current Valuation Multiples ===")
display(p4.multiples_df.T.rename(columns={0: "Value"}))

In [ ]:
# ── DCF sensitivity table ─────────────────────────────────────────────────
if not p4.sensitivity_df.empty:
    print(f"=== DCF Sensitivity Table  (current price: ${current_price:.2f}) ===")
    print("Rows = WACC scenarios  |  Columns = Terminal growth scenarios")
    display(p4.sensitivity_df)
else:
    print("DCF sensitivity table not available (FCF or shares data missing).")

In [ ]:
# ── Historical multiples vs 5Y median ─────────────────────────────────────
if not p4.historical_multiples_df.empty:
    print("=== 5Y Historical Multiples Trend ===")
    display(p4.historical_multiples_df.round(2))
    print("\nCurrent vs 5Y Median:")
    for k, v in p4.multiples_vs_median.items():
        if not np.isnan(v):
            arrow = "⬆ Above" if v > 0.05 else "⬇ Below" if v < -0.05 else "≈ Near"
            print(f"  {k:12s}: {v:+.1%}  {arrow} median")
else:
    print("Historical multiples not available.")

In [ ]:
# ── Valuation decision grid ───────────────────────────────────────────────
mos = p4.margin_of_safety if not (isinstance(p4.margin_of_safety, float) and np.isnan(p4.margin_of_safety)) else 0.0
grid = [
    ("MOS ≥ 20%  + Piotroski ≥ 7  + bullish ≥ 6/11", "Full position, staged 2–3 sessions"),
    ("MOS ≥ 20%  + Piotroski ≥ 7  + bullish 3–5/11", "50% position; add on technical improvement"),
    ("MOS 5–20%  + bullish ≥ 6/11",                   "Partial entry; tight stop"),
    ("MOS ±5% of fair value",                          "At fair value — no asymmetry; hold, no add"),
    ("Overvalued > 15%",                               "Avoid; consider trim if held"),
    ("Altman Z < 1.81",                                "Hard pass — distress risk"),
]
grid_df = pd.DataFrame(grid, columns=["Condition", "Recommended Action"])
print(f"Current situation: MOS = {mos:.1%}  |  Bullish count = {p3.bullish_count}/11")
print(f"Recommendation    : {p4.entry_recommendation}\n")
display(grid_df)

### Phase 4 Commentary

**Four-lens check:**
1. **Relative multiples** — is P/E / EV/EBITDA below own 5Y median and sector median?
2. **DCF** — is the model price above current price (positive MOS)?
3. **Reverse-DCF** — is the market pricing in realistic or heroic growth assumptions?
4. **Quality overlays** — Piotroski ≥ 7 + Altman Z > 2.99 + positive ROIC-WACC spread

A Buy signal requires **≥ 3 of 4 lenses** to agree on direction AND Phase 3 technical count ≥ 4/11.

> ✏️ *Add your valuation notes here.*

---
# Phase 5 — Risk Assessment & Portfolio Context

> **Objective:** Decide whether the stock fits the risk budget and portfolio role —  
> not just whether it looks attractive standalone.
>
> **Gate:** Position sized + stress scenarios documented + portfolio fit classified

**Endpoints used:** `equity.price.historical` (symbol + SPY benchmark)

In [ ]:
p5 = phase5_risk(cfg)
print(f"Portfolio fit           : {p5.portfolio_fit}")
print(f"Recommended position    : {p5.recommended_size:.1%}  (conviction={p5.conviction_size:.1%}, half-Kelly={p5.half_kelly_size:.1%})")
print(f"Gate passed             : {p5.gate_passed}  |  {p5.gate_notes}")

In [ ]:
# ── Full risk KPI table ───────────────────────────────────────────────────
print("=== Risk KPI Summary ===")
display(p5.risk_kpi_df.T.rename(columns={0: "Value"}))

In [ ]:
# ── Regime-conditional beta spotlight ────────────────────────────────────
print("=== Regime-Conditional Beta (Up vs Down Markets) ===")
beta_df = pd.DataFrame([
    {"Metric": "Overall Beta",    "Value": round(p5.beta, 4),      "Interpretation": "> 1 = amplifies market moves"},
    {"Metric": "Up-Market Beta",  "Value": round(p5.beta_up, 4),   "Interpretation": "Participation in rallies"},
    {"Metric": "Down-Market Beta","Value": round(p5.beta_down, 4), "Interpretation": "Amplification in selloffs — lower is better"},
    {"Metric": "Asymmetry",       "Value": round(p5.beta_up - p5.beta_down, 4) if not (np.isnan(p5.beta_up) or np.isnan(p5.beta_down)) else float("nan"),
                                   "Interpretation": "Positive = favourable (more upside, less downside)"},
])
display(beta_df)

In [ ]:
# ── Stress scenarios ──────────────────────────────────────────────────────
print("=== Stress Scenario Estimates ===")
stress_df = pd.DataFrame([
    {"Scenario": k.replace("_", " "), "Est. Impact": f"{v:.1%}"}
    for k, v in p5.stress_scenarios.items()
])
display(stress_df)

In [ ]:
# ── Portfolio fit classification ──────────────────────────────────────────
fit_criteria = [
    ("Core candidate",    "Sharpe > 1.0 · MaxDD < 35% · Down-beta ≤ Up-beta · Calmar > 0.8"),
    ("Satellite",         "Higher growth/MOS but elevated drawdown or tail risk — cap at 2.5%"),
    ("Reject",            "Altman Z < 1.81 · interest coverage < 2× · CVaR < −8%/day"),
]
print(f"Classification result: {p5.portfolio_fit}\n")
display(pd.DataFrame(fit_criteria, columns=["Tier", "Criteria"]))

### Phase 5 Commentary

**Fundamental risk adjustments** (from Phase 2 — apply to position sizing):  
- Accruals Ratio > 15% → reduce size by 0.5%  
- Net Debt/EBITDA > 4× → reduce size by 1%  
- Operating leverage > 2.5× → increase estimated down-beta by 0.2  
- Interest coverage < 2× → satellite only; hard-cap at 2%  

**Final position size = lower of** conviction-based and half-Kelly.

> ✏️ *Add your risk assessment notes here.*

---
# Phase 6 — Market Segment, ETF Benchmark & Peer Relative Analysis

> **Objective:** Prevent single-stock tunnel vision.  
> *"Is this the best way to own exposure to this sector, or are peers offering better risk-adjusted returns?"*
>
> **Gate:** Relative score ≥ 3.5 / 5.0

**Endpoints used:** `equity.price.historical` (universe) · `equity.fundamental.metrics` (per peer)

In [ ]:
p6 = phase6_peer_relative(cfg, p1)
print(f"Sector ETF              : {p6.sector_etf}")
print(f"Information Ratio vs ETF: {p6.information_ratio:.3f}" if not np.isnan(p6.information_ratio) else "Information Ratio: n/a")
print(f"Relative score          : {p6.relative_score:.3f} / 5.0")
print(f"Relative valuation score: {p6.relative_valuation_score:.1f} / 100")
print(f"Rolling 3M rank         : {p6.rolling_3m_rank:.0f}th percentile")
print(f"Gate passed             : {p6.gate_passed}  |  {p6.gate_notes}")

In [ ]:
# ── Relative KPI table ────────────────────────────────────────────────────
print("=== Universe Relative Performance Table ===")
display(p6.relative_table.round(4).sort_values("sharpe", ascending=False))

In [ ]:
# ── Risk-return scatter (bubble = Sharpe) ─────────────────────────────────
rt = p6.relative_table.dropna(subset=["ann_return", "ann_vol", "sharpe"])
if not rt.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    sizes = (rt["sharpe"].clip(lower=0) * 200 + 50).values
    colors = ["red" if idx == cfg.symbol else ("blue" if idx == p6.sector_etf else "grey")
              for idx in rt.index]
    sc = ax.scatter(rt["ann_vol"] * 100, rt["ann_return"] * 100,
                    s=sizes, c=colors, alpha=0.7, edgecolors="white", linewidths=0.5)
    for idx, row in rt.iterrows():
        ax.annotate(idx, (row["ann_vol"]*100, row["ann_return"]*100),
                    fontsize=8, ha="center", va="bottom")
    ax.axhline(0, color="black", lw=0.5, linestyle="--")
    ax.set_xlabel("Annualised Volatility (%)"); ax.set_ylabel("Annualised Return (%)")
    ax.set_title(f"{cfg.symbol} vs Peers — Risk-Return  (bubble size = Sharpe)")
    ax.grid(alpha=0.2)
    plt.tight_layout(); plt.show()
else:
    print("Insufficient data for scatter chart.")

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────
if not p6.corr_matrix.empty and p6.corr_matrix.shape[0] > 1:
    fig, ax = plt.subplots(figsize=(max(6, len(p6.corr_matrix)*0.7), max(5, len(p6.corr_matrix)*0.6)))
    corr = p6.corr_matrix
    im = ax.imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(corr.index)));   ax.set_yticklabels(corr.index, fontsize=8)
    for i in range(len(corr)):
        for j in range(len(corr.columns)):
            val = corr.iloc[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if abs(val) > 0.7 else "black")
    ax.set_title(f"{cfg.symbol} Universe — Daily Return Correlation Heatmap")
    plt.tight_layout(); plt.show()
else:
    print("Insufficient data for correlation heatmap.")

In [ ]:
# ── Peer fundamental quality overlay ─────────────────────────────────────
if not p6.peer_fundamental_df.empty:
    print("=== Peer Fundamental Quality Overlay ===")
    display_cols = [c for c in ["pe_ratio","ev_to_ebitda","roic","free_cash_flow_yield"]
                    if c in p6.peer_fundamental_df.columns]
    display(p6.peer_fundamental_df[display_cols].round(4) if display_cols else p6.peer_fundamental_df.round(4))
    print(f"\nRelative valuation score (target vs peers): {p6.relative_valuation_score:.1f} / 100")
    print("(100 = cheapest quality peer  |  0 = most expensive / lowest quality)")
else:
    print("Peer fundamental data not available.")

### Phase 6 Commentary

**Information Ratio interpretation:**
- IR > 1.0: strong — every 1% of active risk earns > 1% outperformance over the ETF  
- IR 0.5–1.0: good active premium  
- IR < 0.0: the stock underperformed its benchmark ETF for the risk taken — consider holding ETF instead

**5-block scorecard weights:** Return rank 20% · Risk-adjusted 20% · Downside 20% · Consistency 20% · Fundamental quality 20%

> ✏️ *Add your peer comparison notes here.*

---
# Phase 7 — Decision, Execution & Monitoring

> **Objective:** Convert all prior phase outputs into a single actionable, auditable decision —  
> then build the process to manage the position from entry through monitoring to exit.
>
> **Composite weights:** Business Quality 8% · Fundamentals 25% · Technicals 15% · Valuation 20% · Risk 12% · Peer Relative 20%  
> **Action bands:** Strong Buy ≥ 4.2 · Buy ≥ 3.6 · Hold/Watch ≥ 2.8 · Avoid < 2.8

**No new API calls — uses prior phase results only.**

In [ ]:
p7 = phase7_decision(cfg, p1, p2, p3, p4, p5, p6)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    COMPOSITE SCORE & ACTION LABEL                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
label_color = {
    "Strong Buy": "\033[1;32m",  # bold green
    "Buy":        "\033[32m",     # green
    "Hold/Watch": "\033[33m",     # yellow
    "Avoid":      "\033[31m",     # red
}
reset = "\033[0m"
col   = label_color.get(p7.action_label, "")

print("╔══════════════════════════════════════════════════╗")
print(f"║  Symbol  : {cfg.symbol:<39}║")
print(f"║  Score   : {p7.composite_score:<39.4f}║")
print(f"║  Action  : {col}{p7.action_label:<39}{reset}║")
print(f"║  Entry Q : {p7.entry_quality:<39}║")
if p7.hard_override:
    print(f"║  OVERRIDE: {str(p7.hard_override)[:39]:<39}║")
print("╚══════════════════════════════════════════════════╝")

In [ ]:
# ── Score breakdown ───────────────────────────────────────────────────────
weights = {"business_quality":0.08,"fundamentals":0.25,"technicals":0.15,
           "valuation":0.20,"risk_fit":0.12,"peer_relative":0.20}
breakdown_rows = []
for k, w in weights.items():
    s = p7.score_breakdown.get(k, float("nan"))
    breakdown_rows.append({
        "Block":          k.replace("_", " ").title(),
        "Weight":         f"{w:.0%}",
        "Raw Score":      round(s, 3),
        "Weighted":       round(s * w, 4),
        "Bar":            "█" * int(s * 4),
    })
breakdown_df = pd.DataFrame(breakdown_rows)
display(breakdown_df)
print(f"\nComposite: {p7.composite_score:.4f} / 5.0  →  {p7.action_label}")

In [ ]:
# ── Score breakdown radar chart ───────────────────────────────────────────
labels_r = ["Business\nQuality", "Fundamentals", "Technicals", "Valuation", "Risk Fit", "Peer\nRelative"]
values_r = [p7.score_breakdown.get(k, 0) for k in
            ["business_quality","fundamentals","technicals","valuation","risk_fit","peer_relative"]]
values_r += values_r[:1]  # close the polygon

angles = np.linspace(0, 2 * np.pi, len(labels_r), endpoint=False).tolist() + [0]
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.fill(angles, values_r, alpha=0.25, color="steelblue")
ax.plot(angles, values_r, "o-", color="steelblue", lw=2)
ax.set_thetagrids(np.degrees(angles[:-1]), labels_r, fontsize=9)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["1","2","3","4","5"], fontsize=7)
ax.set_title(f"{cfg.symbol} — Phase 7 Score Breakdown\n{p7.action_label} ({p7.composite_score:.3f}/5.0)",
             pad=15, fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Execution plan ────────────────────────────────────────────────────────
print("=== Execution Plan ===")
exec_data = [
    ("Current price",       f"${current_price:.2f}"),
    ("ATR(14)",              f"${p7.atr_stop + current_price - current_price:.2f}" if False else f"${p3.atr:.4f}"),
    ("Stop price (2×ATR)",  f"${p7.atr_stop:.2f}"),
    ("Risk per share (R)",  f"${p7.risk_per_share:.2f}"),
    ("Target 1 (1R)",       f"${p7.target_1r:.2f}  → move stop to break-even"),
    ("Target 2 (2R)",       f"${p7.target_2r:.2f}  → take 50% off position"),
    ("Target 3 (3R)",       f"${p7.target_3r:.2f}  → trail remainder with 1.5×ATR"),
    ("Entry quality",       p7.entry_quality),
    ("Time stop date",      p7.time_stop_date),
]
display(pd.DataFrame(exec_data, columns=["Parameter", "Value"]))

In [ ]:
# ── Staged entry tranches ─────────────────────────────────────────────────
print("=== Staged Entry Tranches ===")
tranche_notes = {
    "tranche_1": "Initial entry — open now if signal confirmed",
    "tranche_2": "Add if thesis confirms (VWAP hold / technicals improve / 1R not hit)",
    "tranche_3": "Final add after 1R gain achieved",
}
tranche_df = pd.DataFrame([
    {"Tranche": k, "Size (% of target)": f"{v:.0%}", "Condition": tranche_notes.get(k, "")}
    for k, v in p7.staged_entry.items()
])
display(tranche_df)

In [ ]:
# ── Thesis invalidation framework ─────────────────────────────────────────
print("=== Thesis Invalidation Triggers ===")
triggers = [
    ("Revenue miss",         f"> {abs(p7.monitoring_triggers['revenue_miss_threshold']):.0%} below consensus for 2 consecutive quarters"),
    ("Gross margin decline", f"Gross margin falls by > {abs(p7.monitoring_triggers['gross_margin_decline']):.0%} in any quarter"),
    ("Share dilution",       f"Net dilution > {p7.monitoring_triggers['dilution_12m']:.0%} in any 12-month period"),
    ("MOS turns negative",   "DCF fair value drops below current price"),
    ("Weekly death cross",   "Weekly SMA50 crosses below weekly SMA200"),
    ("Peer Sharpe rank",     f"Falls below {p7.monitoring_triggers['peer_sharpe_rank_threshold']:.0%} peer percentile for 2 consecutive 60-day windows"),
    ("Management integrity", "SEC filing restatement or key executive departure without succession"),
]
display(pd.DataFrame(triggers, columns=["Thesis Pillar", "Invalidation Event"]))
print()
if p7.monitoring_triggers.get("earnings_note", "OK") != "OK":
    print(f"⚠  {p7.monitoring_triggers['earnings_note']}")
if p7.monitoring_triggers.get("ir_note", "OK") != "OK":
    print(f"⚠  {p7.monitoring_triggers['ir_note']}")

In [ ]:
# ── Monitoring cadence ────────────────────────────────────────────────────
print("=== Monitoring Cadence ===")
cadence = [
    ("Quarterly", "After each earnings",  "Thesis KPI set (revenue, margins, FCF, accruals)     → 2 consecutive declines: reduce to 50% size"),
    ("Weekly",    "Ongoing",              "Technical regime (SMA50/200 weekly, ADX)             → Weekly death cross: exit half"),
    ("Monthly",   "Rolling",              "Peer relative rank (Sharpe, Information Ratio)       → Below peer median 2 months: reduce to satellite"),
    ("Monthly",   "Rolling",              "Valuation gap vs DCF fair value                      → MOS < 5% or negative: trim to 50%"),
    ("Daily",     "Continuous",           "Risk budget usage                                    → Any tranche > 5% of portfolio: rebalance"),
    ("Weekly",    "Ongoing",              "Earnings date proximity                              → Within 5 days: review sizing"),
    ("Weekly",    "After 1R gain",        "Stop distance vs ATR                                 → After 1R: move stop to break-even"),
]
display(pd.DataFrame(cadence, columns=["Frequency", "Trigger Timing", "KPI  →  Action"]))

In [ ]:
# ── Handoff template ──────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║                    POSITION HANDOFF TEMPLATE                         ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
h = p7.handoff
print(f"\n1. Investment Thesis:")
print(f"   {h['investment_thesis']}")
print(f"\n2. Top Bullish Drivers:")
for i, d in enumerate(h['bullish_drivers'], 1):
    print(f"   {i}. {d}")
print(f"\n3. Invalidation Events:")
for i, e in enumerate(h['invalidation_events'], 1):
    print(f"   {i}. {e}")
print(f"\n4. Fair Value Range & MOS:")
fv = h['fair_value_range']
print(f"   DCF fair value: ${fv['dcf']:.2f}  |  MOS: {fv['margin_of_safety']:.1%}  |  Verdict: {fv['valuation_verdict']}")
print(f"\n5. Peer-Relative Verdict:")
pr = h['peer_relative']
print(f"   Information Ratio: {pr['information_ratio']:.3f}  |  Relative Score: {pr['relative_score']:.2f}/5.0")
print(f"   {pr['peer_verdict']}")
print(f"\n6. Trade Plan:")
tp = h['trade_plan']
print(f"   Composite: {tp['composite_score']:.3f}  →  {tp['action_label']}")
print(f"   Entry quality: {tp['entry_quality']}  |  Portfolio fit: {tp['portfolio_fit']}")
print(f"   ATR: {tp['atr']:.4f}  |  Stop: ${p7.atr_stop:.2f}  |  2R target: ${p7.target_2r:.2f}")
print(f"   Time stop: {p7.time_stop_date}")

---
### Phase 7 Commentary & Final Assessment

**Hard override rules (automatically applied):**
- Altman Z < 1.81 → forced **Avoid** (distress risk)
- Accruals Ratio > 20% → capped at **Hold/Watch**
- Weekly trend bearish → technical block capped at 2.0/5.0
- Earnings within 5 days → technical block capped at 2.5/5.0
- IR vs ETF < 0 AND relative score < 3.0 → flag ETF as superior alternative

**Staged entry rationale:**  
Never enter a full position in a single order. Building in 3 tranches reduces average entry cost if the stock dips after the first tranche and prevents committing full capital before the setup has confirmed.

**Time stop rationale:**  
If the stock has not moved in the direction of the thesis within 63 calendar days (1 earnings cycle) of entry, exit regardless of whether the price stop has been hit. This prevents capital being trapped in "dead money" positions.

> ✏️ *Add your final investment decision notes here.*

---
*Notebook generated by `Analysis/stock_analysis.py` v1.0 · Provider: `fmp_cached` · Spec: `PHASED_ANALYSIS_MASTER_PLAN.md` v1.1*